In [1]:
import os
import itertools
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import igraph as ig
import leidenalg as la
import random
from skleSSSarn.metrics import jaccard_score
from tabulate import tabulate
from IPython.display import HTML, display
import html
import time
from sklearn.metrics import normalized_mutual_info_score

### reading

In [2]:
with open("oracle-cards-20260508090243.json", "r", encoding="utf-8") as file:
    FullCardList = json.load(file)

# remove add duplicates explained below ======================================================================
before = len(FullCardList)
print("Total Before Removal: ", before)

itemsRemoved = {}

for c in FullCardList[:]: # copy so we can delete and iterate at same time
    if "//" in c["name"]:
        parts=c["name"].split("//")
        if parts[0].strip() == parts[1].strip():
            itemsRemoved[c["name"]] = c.items()
            FullCardList.remove(c)

print("number of items removed (//): ", len(itemsRemoved))
print("")
print("Total after removal: ", len(FullCardList))
print("Total number of items removed: ", before - (len(FullCardList)))

removeElements = ["set_type",
                  "mtgo_id",
                  "tcgplayer_id",
                  "lang",
                  "released_at",
                  "uri",
                  "scryfall_uri",
                  "highres_image",
                  "image_status",
                  "image_uris",
                  "set_uri",
                  "set_search_uri",
                  "scryfall_set_uri",
                  "rulings_uri",
                  "prints_search_uri",
                  "watermark",
                  "artist",
                  "artist_ids",
                  "illustration_id",
                  "border_color",
                  "frame",
                  "frame_effects",
                  "security_stamp",
                  "full_art",
                  "testless",
                  "booster",
                  "story_spotlight",
                  "related_uris",
                  "purchase_uris",
                  "digital",
                  "foil",
                  "nonfoil",
                  "preview",
                  "games",
                  "finishes",
                  "oversized",
                  "promo",
                  "reprint",
                  "textless"]

for card in FullCardList:
    for key in removeElements:
        card.pop(key,None)

cardsInfo = {card["name"]: card for card in FullCardList}

Total Before Removal:  37442
number of items removed (//):  2189

Total after removal:  35250
Total number of items removed:  2192


In [3]:
#
# read the text files
# get out the deck list a set of unqinue cards
# and add the cards to the complete snapshot set
#
def readFiletxt(file, cardSet):

    decklist = set()
    
    with open(file, "r", encoding="cp1252") as f:
        for line in f:
            
            if(line.strip() == "Sideboard"):
                break
            else:
                parts = line.split(" ",1)
                #print(parts[1])
                if ( checkLands(parts[1].strip()) ):
                    continue

                else:
                    cardSet.add(parts[1].strip())
                    decklist.add(parts[1].strip())
        
    return decklist, cardSet


#
# read the mwDeck files
# get out the deck list a set of unqinue cards
# and add the cards to the complete snapshot set
#

def readFilemwDeck(file, cardSet):
    decklist = set()
    
    with open(file, "r", encoding="cp1252") as f:
        for line in f:
            #print(line)

            if line.startswith("//"):
                continue

            if line.startswith("SB:"):
                continue
            
            else:
                parts = line.split(" ",2)
                #print(parts[2].strip())
                if ( checkLands(parts[2].strip()) ):
                    continue

                else:
                    cardSet.add(parts[2].strip())
                    decklist.add(parts[2].strip())
        
    return decklist, cardSet

In [4]:
#
# using this method to check cards and see if they are lands
# land cards act as bridges to archtypes that share colour but they themselves dont carry much information,
# since they are required to be in decks they lead to archtypes being condensed together
#
#
def checkLands(checkCard):

    typeLineCheck = ""
    pCards = []

    if "/" in checkCard:
        parts=checkCard.split("/")
        checkCard=parts[0]
        

    # if the card name flat out exists in the dict save it
    if checkCard in cardsInfo:
            typeLineCheck = cardsInfo[checkCard]["type_line"]

    else: # esle try and search for similar
        for c in FullCardList:
            if checkCard.lower() in c["name"].lower():
                pCards.append(c["name"])

        # if only one similar save it
        if (len(pCards) == 1):
            typeLineCheck = cardsInfo[pCards[0]]["type_line"]
                
        
        else:
            #print("+++++++++++++++++++++++++++++++++++++++")
            #print("")
            #print("Error")
            #print("")
            #print(checkCard)
            #print("")
            #print(pCards)
            #print("")
            #print("+++++++++++++++++++++++++++++++++++++++")
            
            #if a card name appears multiple times high likelihood of not being a land
            return False


    if ("Land" in typeLineCheck) and ("//" not in typeLineCheck):
        return True

    else:
        return False
    

In [5]:
#
# this goes into my folders to grab the specific files
# based on file type it uses the correct file reader
# then it prints out the number of total cards and the number of decks
# number of decks used for normalisation
# number of unqiue cards is number of nodes for this snapshot
def getSnapshot(folder):

    snapshotSet = set()
    listOfDecks = []

    for itemMajor in os.listdir(folder):
        #print(itemMajor)
        newpath = os.path.join(folder, itemMajor)
        for itemMinor in os.listdir(newpath):
            #print(itemMinor)
            newestpath = os.path.join(newpath, itemMinor)

            if newestpath.endswith(".mwDeck"):
                temp, snapshotSet = readFilemwDeck(newestpath,snapshotSet)
                listOfDecks.append(temp)
                

            if newestpath.endswith(".txt"):
                temp, snapshotSet = readFiletxt(newestpath,snapshotSet)
                listOfDecks.append(temp)
                


    print("-- Statisitcs --")
    print("   Snapshot               :" , folder)
    print("   number of cards        :" , len(snapshotSet))
    print("   number of Decks        :" , len(listOfDecks))
    print("")
    return snapshotSet, listOfDecks
                
    

In [6]:
##
# use sorted tuples for dictionary to form weighted edge list
# take list of unique cards in each deck
# increase dicitonary value for each time a pair appears
# return this dictionary and a dictionary of each time a card appears
#

def makeEdgeList(decks):

    countPairs = {} #dicitonary of pairs of cards (SORT ALPHABETICALLY) count of pairs appearance
    countCards = {} #dict of cards appearance 

    for deck in decks:
        for card in deck:
            if card not in countCards:
                countCards[card] = 1

            else:
                countCards[card] += 1


        for pairs in itertools.combinations(deck,2):
            pairs = tuple(sorted(pairs))
            
            if pairs not in countPairs:
                countPairs[pairs] = 1

            else:
                countPairs[pairs] += 1

    return countCards, countPairs
    

In [7]:
#
#
# make graphs
# make an undirected weighted graph 
#
def makeUndirectedGraph(edges, noDecks):

    G = nx.Graph()
    for key in edges:

        G.add_edge(key[0],key[1],weight = edges[key]/noDecks) # edges are normalised by => pair appearances /total number of decks

    return G

#
# directed weighted graph
#
def makeDirectedGraph(edges, cards):

    G = nx.DiGraph()
    for key in edges:

        # (from_node , to_node) 
        # probablity of to_node, given from_node
        temp = edges[key]/ cards[key[0]]
        G.add_edge(key[0],key[1],weight=temp)

        temp = edges[key]/ cards[key[1]]
        G.add_edge(key[1],key[0],weight=temp)

    return G
    

In [8]:

#
# print out the useful and main statistics of the graph
# prints out information based on graph type (directed/undirected)
#
def graphStats(G):

    # universal stats =========================================================
    noNodes = G.number_of_nodes()
    noEdges = G.number_of_edges()
    density = nx.density(G)

    print("number of Nodes                          : " , noNodes)
    print("number of Edges                          : " , noEdges)
    print("density (fraction of all possible nodes) : " , density)

    # undirected stats ===================================================
    if not nx.is_directed(G): #undirected 
        avgDegree = 0
        for node, deg in G.degree:
            avgDegree += deg

        avgDegree = avgDegree/noNodes
        print("Average degree                           : ", avgDegree)

        
        avgEdgeW = 0
        temp = G.edges(data=True)
        for n in temp:
            avgEdgeW += n[2]["weight"]

        avgEdgeW = avgEdgeW/noEdges
        print("Average Edge Weight                      : ", avgEdgeW)

        


    #directed stats===========================================
    else:
        avgInDegree = 0
        for node, value in G.in_degree():
            avgInDegree += value

        avgInDegree = avgInDegree/noNodes
        print("Average In degree                         : ", avgInDegree)
        
        avgOutDegree = 0
        for node, value in G.out_degree():
            avgOutDegree += value

        avgOutDegree = avgOutDegree/noNodes
        print("Average Out degree                        : ", avgOutDegree)
    

    #number of connected components
    
    print("")
    print("")
    return noNodes
    

In [9]:
def snapToGraph(pathway, cardSetSoFar):

    cardsInSnap = set()

    #parse the snapshot and get a set of all the cards
    # and a list of sets of all the decks
    # i.e each deck but only the unique cards in said deck
    cardset, decks = getSnapshot(pathway)

    #get card occurances
    #get pairwise occurances
    cardsOcc, pairsOcc = makeEdgeList(decks)


    #return the updated cards set
    cardSetSoFar.update(cardset) # this is so we have a set which can be built upon and will contain all cards period
    cardsInSnap.update(cardset) # this is so we have a set of exclusively this snapshot
    #print("length : ", len(decks))
    print("# cards that have appeared so far :", len(cardSetSoFar))

    graph = makeUndirectedGraph(pairsOcc, len(decks) )
    digraph = makeDirectedGraph(pairsOcc, cardsOcc)

    print("")
    print("------------- Undirected Graph Stats -------------")
    graphStats(graph)
    comms, commsDict, modularity = getlouvainPart2(graph, cardsOcc, 25, 0.2)


    print("")
    print("------------- Directed Graph Stats -------------")
    graphStats(digraph)


    
    cardsOccNorm = cardsOcc.copy()
    for name,x in cardsOccNorm.items():
        cardsOccNorm[name]= x /len(decks)

    # graph
    # digraph = directed graph
    # cardSetSoFar =  continuous set of cards that will be added to with each subsequent snapshot
    # cardsOcc = dictionary of cards and that cards number of occurances
    # commsDict = dictionary of communities
    # cardsInSnap = set of cards in this specific snapshot
    # cardsPageRank = dict of cards in this snpashot and thier pagerank
    # cardsOccNorm = dict of cards in this snpashot and thier occurance normalised
    return graph, digraph, cardSetSoFar, cardsOcc, commsDict, cardsInSnap, cardsOccNorm, modularity

### get graph to test

In [12]:
totalcards = set()

cardOccurancesNorm = []
modularity = []
numberOfCommunities = []
g1, diG1, totalcards, cardOccurances1, communityList1, snap1Cards, cardOccurancesNorm1, mod1 = snapToGraph("decklists\\Snapshot 1", totalcards)

cardOccurancesNorm.append(cardOccurancesNorm1)
modularity.append(mod1)
numberOfCommunities.append(len(communityList1))

-- Statisitcs --
   Snapshot               : decklists\Snapshot 1
   number of cards        : 403
   number of Decks        : 697

# cards that have appeared so far : 403

------------- Undirected Graph Stats -------------
number of Nodes                          :  403
number of Edges                          :  5852
density (fraction of all possible nodes) :  0.07224423786773329
Average degree                           :  29.042183622828784
Average Edge Weight                      :  0.011863410319197694


Number of communities :  10
Modularity Score      :  0.6068814711075714

------------- Directed Graph Stats -------------
number of Nodes                          :  403
number of Edges                          :  11704
density (fraction of all possible nodes) :  0.07224423786773329
Average In degree                         :  29.042183622828784
Average Out degree                        :  29.042183622828784




In [18]:
g2, diG2, totalcards, cardOccurances2, communityList2, snap2Cards, cardOccurancesNorm2, mod2 = snapToGraph("decklists\\Snapshot 2", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 2
   number of cards        : 415
   number of Decks        : 986

# cards that have appeared so far : 565

------------- Undirected Graph Stats -------------
number of Nodes                          :  415
number of Edges                          :  6530
density (fraction of all possible nodes) :  0.07601420173447412
Average degree                           :  31.46987951807229
Average Edge Weight                      :  0.011464173777449238


Number of communities :  7
Modularity Score      :  0.5371515585980366

------------- Directed Graph Stats -------------
number of Nodes                          :  415
number of Edges                          :  13060
density (fraction of all possible nodes) :  0.07601420173447412
Average In degree                         :  31.46987951807229
Average Out degree                        :  31.46987951807229




In [30]:
g3, diG3, totalcards, cardOccurances3, communityList3, snap3Cards, cardOccurancesNorm3, mod3 = snapToGraph("decklists\\Snapshot 3", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 3
   number of cards        : 213
   number of Decks        : 210

# cards that have appeared so far : 594

------------- Undirected Graph Stats -------------
number of Nodes                          :  213
number of Edges                          :  2289
density (fraction of all possible nodes) :  0.10138187616263619
Average degree                           :  21.492957746478872
Average Edge Weight                      :  0.029742661590631488


Number of communities :  5
Modularity Score      :  0.6222757942327986

------------- Directed Graph Stats -------------
number of Nodes                          :  213
number of Edges                          :  4578
density (fraction of all possible nodes) :  0.10138187616263619
Average In degree                         :  21.492957746478872
Average Out degree                        :  21.492957746478872




In [38]:
g4, diG4, totalcards, cardOccurances4, communityList4, snap4Cards, cardOccurancesNorm4, mod4 = snapToGraph("decklists\\Snapshot 4", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 4
   number of cards        : 423
   number of Decks        : 325

# cards that have appeared so far : 704

------------- Undirected Graph Stats -------------
number of Nodes                          :  423
number of Edges                          :  5764
density (fraction of all possible nodes) :  0.0645804622813799
Average degree                           :  27.252955082742318
Average Edge Weight                      :  0.014482997918111322


Number of communities :  8
Modularity Score      :  0.6134561932336581

------------- Directed Graph Stats -------------
number of Nodes                          :  423
number of Edges                          :  11528
density (fraction of all possible nodes) :  0.0645804622813799
Average In degree                         :  27.252955082742318
Average Out degree                        :  27.252955082742318




In [44]:
g5, diG5, totalcards, cardOccurances5, communityList5, snap5Cards, cardOccurancesNorm5, mod5 = snapToGraph("decklists\\Snapshot 5", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 5
   number of cards        : 211
   number of Decks        : 184

# cards that have appeared so far : 752

------------- Undirected Graph Stats -------------
number of Nodes                          :  211
number of Edges                          :  2153
density (fraction of all possible nodes) :  0.09717896637327916
Average degree                           :  20.407582938388625
Average Edge Weight                      :  0.03201044043700466


Number of communities :  5
Modularity Score      :  0.6689417651925353

------------- Directed Graph Stats -------------
number of Nodes                          :  211
number of Edges                          :  4306
density (fraction of all possible nodes) :  0.09717896637327916
Average In degree                         :  20.407582938388625
Average Out degree                        :  20.407582938388625




In [31]:
totalcards = set()
g6, diG6, totalcards, cardOccurances6, communityList6, snap6Cards, cardOccurancesNorm6, mod6 = snapToGraph("decklists\\Snapshot 6", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 6
   number of cards        : 349
   number of Decks        : 542

# cards that have appeared so far : 349

------------- Undirected Graph Stats -------------
number of Nodes                          :  349
number of Edges                          :  4236
density (fraction of all possible nodes) :  0.0697559529690742
Average degree                           :  24.275071633237822
Average Edge Weight                      :  0.016626072776307937


Number of communities :  6
Modularity Score      :  0.666155143991649

------------- Directed Graph Stats -------------
number of Nodes                          :  349
number of Edges                          :  8472
density (fraction of all possible nodes) :  0.0697559529690742
Average In degree                         :  24.275071633237822
Average Out degree                        :  24.275071633237822




In [12]:
totalcards = set()
g7, diG7, totalcards, cardOccurances7, communityList7, snap7Cards, cardOccurancesNorm7, mod7 = snapToGraph("decklists\\Snapshot 7", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 7
   number of cards        : 524
   number of Decks        : 889

# cards that have appeared so far : 524

------------- Undirected Graph Stats -------------
number of Nodes                          :  524
number of Edges                          :  8729
density (fraction of all possible nodes) :  0.06370323880139536
Average degree                           :  33.31679389312977
Average Edge Weight                      :  0.008171693053203512


Number of communities :  8
Modularity Score      :  0.6001552677731691

------------- Directed Graph Stats -------------
number of Nodes                          :  524
number of Edges                          :  17458
density (fraction of all possible nodes) :  0.06370323880139536
Average In degree                         :  33.31679389312977
Average Out degree                        :  33.31679389312977




In [18]:
totalcards = set()
g8, diG8, totalcards, cardOccurances8, communityList8, snap8Cards, cardOccurancesNorm8, mod8 = snapToGraph("decklists\\Snapshot 8", totalcards)

-- Statisitcs --
   Snapshot               : decklists\Snapshot 8
   number of cards        : 410
   number of Decks        : 678

# cards that have appeared so far : 410

------------- Undirected Graph Stats -------------
number of Nodes                          :  410
number of Edges                          :  6267
density (fraction of all possible nodes) :  0.07474506529906375
Average degree                           :  30.570731707317073
Average Edge Weight                      :  0.011177149775030955


Number of communities :  5
Modularity Score      :  0.5409539722539107

------------- Directed Graph Stats -------------
number of Nodes                          :  410
number of Edges                          :  12534
density (fraction of all possible nodes) :  0.07474506529906375
Average In degree                         :  30.570731707317073
Average Out degree                        :  30.570731707317073




### consensus for testing

In [10]:

#### consensus cluster Louvain
#
#
#
#

def makeUndirectedGraph2(edges, noDecks, all_nodes):

    G = nx.Graph()
    G.add_nodes_from(all_nodes)
    for key in edges:

        G.add_edge(key[0],key[1],weight = edges[key]/noDecks) # edges are normalised by => pair appearances /total number of decks

    return G


# main body of consensus
def getlouvain(G):
     return community_louvain.best_partition(G)
# function to call louvain and then take the partition and build up the adjacency matrix

def consensusLouvain(G, noTimes, threshold, iterations):
    going = True
    output = 0
    countIters = 0
    while(going):

        
        tempEL = {}
        #apply louv to G a number of times
        for n in range(noTimes):
            tempP = getlouvain(G)

            #build a matrix from this continuousley adding 
            for pairs in itertools.combinations(tempP.keys(),2):
                
                pairs = tuple(sorted(pairs))
                #if both nodes are in the same partition
                if tempP[pairs[0]] == tempP[pairs[1]]:
                    if pairs not in tempEL:
                        tempEL[pairs] = 1

                    else:
                        tempEL[pairs] += 1


        if (all(v==0 or v==noTimes for v in tempEL.values())):
            going = False

        tempEL = {pair: count for pair, count in tempEL.items() if count >= (threshold*noTimes)}
        G = makeUndirectedGraph2(tempEL, noTimes, G.nodes())

        
        
        output = getlouvain(G)

        
        if countIters > iterations - 1:
            going = False
        else:
            countIters += 1
            #print(countIters)

        

    finalpartition = output
    return finalpartition
    
    
#main body where all the usefull information is printed out post consensus
def getlouvainPart2(G, cards, numberOfTimes, threshold):
    #random.seed(seed)

    
    
    #p = community_louvain.best_partition(G, randomize = r)
    p = consensusLouvain(G, numberOfTimes, threshold, 10)


    # ========== finalised scores and communities ==========
    outputDict = {}
    print("Number of communities : " , len(set(p.values())))
    tempScore = community_louvain.modularity(p, G)
    print("Modularity Score      : " ,  tempScore)
    #print("")

    for i in set(p.values()):
        temp = []

        for node in p:
            if p[node] == i:
                temp.append((node, cards[node]))


        #print("  ===== Communitiy ", i ," =====")

        sortedD = sorted(temp, key= lambda x: x[1], reverse=True)
        #print(len(sortedD))
                
        if len(sortedD) > 10:
            x = 0
            while (x < 10):
                #print(sortedD[x])
                x +=1

        if len(sortedD) <= 10:
            x = 0
            while (x < len(sortedD)):
                #print(sortedD[x])
                x +=1

        outputDict["com" + str(i)] = sortedD
        #print("")
        #print("")
        #print("")
    return p , outputDict, tempScore
    
    #so run louvain 100s of times
#store results in a matrix
#build a graph using said matrics with the edges between them being Aij / total runs

In [19]:

# this is a major important run
# here wihtin the same parameters consensus louvain is ran 5 times to ensure 
# it has a stable output
#
#
def confirmStable(noTimes,threshold):
    previous = []
    scores = []

    for x in range(5):
    
    
    
        out1, out2, out3 = getlouvainPart2(g8, cardOccurances8, noTimes, threshold)

        if(len(out1) == len(g8.nodes()) ):

            if len(previous) >= 1:
                for index,y in enumerate(previous):
                    nodes = sorted(y.keys())
                    compare1 = [y[n] for n in nodes]
                    compare2 = [out1[n] for n in nodes]
                    temporary = normalized_mutual_info_score(compare1, compare2)

                    print(f"     compare run {index} vs run {x} :" , temporary)
                    scores.append(temporary)
                
                      
            previous.append(out1)
            print("")
            print("")
        else:
            print("")
            print("")
            print("===================================================================================")
            print("error")    
            print("===================================================================================")

    finalAvg = sum(scores) / len(scores)
    print(finalAvg)
    return finalAvg, out1



def majorTest():
    rangeOfThres = [ 0.2, 0.3 , 0.4, 0.5]
    rangeOfTimes = [50, 100, 200]
    checkAcrossParams = {}

    allTheOutputs = []

    ### run each param pair
    for x in rangeOfThres:
        for y in rangeOfTimes:

            #test within the parameter
            finalAvg, partitionToCheck = confirmStable(y,x)
            allTheOutputs.append((y,x,finalAvg,partitionToCheck))


    for i, j in itertools.combinations(allTheOutputs, 2):
        timeI, threshldI, avgI, checkI = i
        timeJ, threshldJ, avgJ, checkJ = j

        nodes = sorted(checkI.keys())
        compare1 = [checkI[n] for n in nodes]
        compare2 = [checkJ[n] for n in nodes]

        temporary = normalized_mutual_info_score(compare1, compare2)
        
        checkAcrossParams[((timeI, threshldI),(timeJ, threshldJ))] = temporary
    
                
    ####
    return allTheOutputs, checkAcrossParams
    
finalout1, finalout2 = majorTest()

Number of communities :  5
Modularity Score      :  0.5409925554431647


Number of communities :  5
Modularity Score      :  0.5409539722539107
     compare run 0 vs run 1 : 0.9804062618896612


Number of communities :  5
Modularity Score      :  0.5410842313070054
     compare run 0 vs run 2 : 0.9512392007653708
     compare run 1 vs run 2 : 0.970813883023505


Number of communities :  5
Modularity Score      :  0.5410383304175923
     compare run 0 vs run 3 : 0.9371812338871138
     compare run 1 vs run 3 : 0.936968550477634
     compare run 2 vs run 3 : 0.9661602557656634


Number of communities :  6
Modularity Score      :  0.5464139783503951
     compare run 0 vs run 4 : 0.972066488614345
     compare run 1 vs run 4 : 0.9530147395293036
     compare run 2 vs run 4 : 0.9246799970828147
     compare run 3 vs run 4 : 0.911029544621143


0.9503560155656554
Number of communities :  5
Modularity Score      :  0.5409539722539106


Number of communities :  5
Modularity Score      :  0.540

In [22]:
finalout2

{((50, 0.2), (100, 0.2)): np.float64(0.9538881067007982),
 ((50, 0.2), (200, 0.2)): np.float64(0.9246799970828147),
 ((50, 0.2), (50, 0.3)): np.float64(0.953888106700798),
 ((50, 0.2), (100, 0.3)): np.float64(0.981471361106905),
 ((50, 0.2), (200, 0.3)): np.float64(0.953888106700798),
 ((50, 0.2), (50, 0.4)): np.float64(0.9400005038110766),
 ((50, 0.2), (100, 0.4)): np.float64(0.9674952425159837),
 ((50, 0.2), (200, 0.4)): np.float64(0.9814713611069049),
 ((50, 0.2), (50, 0.5)): np.float64(0.9635817977899384),
 ((50, 0.2), (100, 0.5)): np.float64(0.9635817977899385),
 ((50, 0.2), (200, 0.5)): np.float64(0.9674952425159837),
 ((100, 0.2), (200, 0.2)): np.float64(0.9720922270826892),
 ((100, 0.2), (50, 0.3)): np.float64(0.9999999999999999),
 ((100, 0.2), (100, 0.3)): np.float64(0.9723997043941911),
 ((100, 0.2), (200, 0.3)): np.float64(0.9999999999999999),
 ((100, 0.2), (50, 0.4)): np.float64(0.9861186215506471),
 ((100, 0.2), (100, 0.4)): np.float64(0.9584291548126425),
 ((100, 0.2), (2

In [23]:
for x in finalout1:
    print(x[0], x[1], x[2])

50 0.2 0.9503560155656554
100 0.2 0.9638703298873323
200 0.2 0.9736426284979027
50 0.3 0.9561148262836439
100 0.3 0.9637090519873663
200 0.3 0.9774811463687637
50 0.4 0.9750731913232886
100 0.4 0.9750574928875857
200 0.4 0.9751074539367932
50 0.5 0.9852799627425872
100 0.5 0.9797848986482227
200 0.5 0.9805916862863775
